# Experiment 02 — accepted-dollar caps

Parent: [01-initial.ipynb](01-initial.ipynb), using `hyper-ai-v11.py`, the same guard record, full indicator history and execution fixes.

## Question and change

Does preventing capacity-driven redistribution improve v11's returns and risk? Compare the parent with one predeclared treatment over **both original windows**: 1 January–5 October 2026 and 1 March 2023–5 October 2026 (ends exclusive). Each of the four runs starts independently with $150,000 and uses the same two-day decision schedule, six-vault limit, 33% concentration limit and 33% vault-TVL limit.

For each selected vault, first calculate its normalised inverse-risk share of the original fresh deployable budget. Its accepted dollars are the minimum of that original request, the concentration cap and the unchanged TVL-size-risk limit. Unallocated capital remains in cash. Explicit targets survive later normalisation; pinned non-redeemable holdings retain the parent's treatment. Source ranking, selection, gates, hysteresis, fee accounting and trade thresholds remain the same. Holdings can diverge over time because sizing changes each run's portfolio state.

The research harness changes only the AlphaModel factory during a decision and restores it afterwards. A same-state parent sizing counterfactual in each treatment cycle makes redistribution measurable. Both parent controls must reproduce experiment 01. The allocation ledger verifies that accepted and final targets never exceed the original request. The experiment changes sizing rather than adding a risk veto or changing the selected-vault limit.

The engine rounds the long start down to the cash-only 28 February 2023 cycle. The dataset uses today's selected universe and static guard permissions; historical deposit-state snapshots begin on 11 April 2026. Earlier deposits are assumed open. Those historical-universe limitations remain applicable.

## Restored-data results — 7 October 2026

Accepted-dollar caps still underperform: 9.31% CAGR versus 41.20% for the parent, with 51.32% mean cash versus 8.83%. Parent controls reproduce notebook 01 on both periods; the original-request allocation assertions pass.

### Strategy comparison — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fresh engine backtests on the same pinned publication and merged readers, each starting with $150,000. Full-history runs are retained separately in the saved outputs.

| Strategy | Final equity (USD) | CAGR | Sharpe | Maximum drawdown | Trades | Mean cash |
| --- | --- | --- | --- | --- | --- | --- |
| Short — v11 parent | \$194,520.81 | 41.20% | 1.96 | -6.40% | 581 | 8.83% |
| Short — accepted-dollar cap | \$160,350.69 | 9.31% | 1.31 | -4.22% | 474 | 51.32% |

Both periods completed. Historical benchmark differences are separate from mandatory equality between controls on identical inputs. See [the rerun report](restored-data-rerun-2026-10-07.md) for input and clock provenance.


### Restored leader history rerun — 7 October 2026

This execution uses the freshly downloaded, pinned repaired dataset and the merged leader-share and recorded-cap readers. Strategy settings, searches and period boundaries are unchanged. Historical benchmark differences are displayed separately from mandatory source/control equality on the same inputs. Generated results are isolated under `_artifacts-restored-2026-10-07/02`.


In [1]:
import datetime
from copy import deepcopy
import hashlib
import importlib.util
import logging
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from dotenv import load_dotenv

from tradingstrategy.client import Client
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.execution_context import notebook_execution_context
from tradeexecutor.strategy.parameters import StrategyParameters
from tradeexecutor.strategy.reserve_currency import ReserveCurrency
from tradeexecutor.statistics.key_metric import (
    calculate_cagr,
    calculate_max_drawdown,
    calculate_sharpe,
    calculate_sortino,
)
from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode

load_dotenv(override=True)
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)

STRATEGY_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v11.py")
GUARD_PATH = Path("/Users/moo/code/strategies/deploy/hyper-ai-v10-new-guard-vault-info.json")
os.environ["HYPER_AI_GUARD_CONFIG_FILE"] = str(GUARD_PATH)
assert STRATEGY_PATH.is_file(), STRATEGY_PATH
assert GUARD_PATH.is_file(), GUARD_PATH

spec = importlib.util.spec_from_file_location("hyper_ai_v11", STRATEGY_PATH)
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)
source_sha256 = hashlib.sha256(STRATEGY_PATH.read_bytes()).hexdigest()
parameters = StrategyParameters.from_class(strategy.Parameters)
client = Client.create_jupyter_client()

display(pd.DataFrame([
    {"Input": "Strategy source", "Value": str(STRATEGY_PATH)},
    {"Input": "Strategy SHA-256", "Value": source_sha256},
    {"Input": "Guard record", "Value": str(GUARD_PATH)},
    {"Input": "Decision cycle", "Value": str(parameters.cycle_duration)},
    {"Input": "Initial equity (USD)", "Value": parameters.initial_cash},
]))

RESEARCH_DIR = Path("/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
sys.path.insert(0, str(RESEARCH_DIR))
from accepted_dollar_cap import AuditedAlphaModel, make_decide_trades, calculate_position_profit_curves
from tradeexecutor.analysis.weights import calculate_asset_weights
from tqdm_loggable.auto import tqdm

harness_sha256 = hashlib.sha256((RESEARCH_DIR / "accepted_dollar_cap.py").read_bytes()).hexdigest()
display(pd.DataFrame([
    {"Input": "Parent notebook", "Value": "01-initial.ipynb"},
    {"Input": "Sizing harness SHA-256", "Value": harness_sha256},
    {"Input": "Treatment", "Value": "Original requested dollars; no redistribution after caps"},
    {"Input": "Max vaults / concentration / TVL share", "Value": "6 / 33% / 33%"},
]))
from score_matrix_visualisation import with_backtest_period

def backtest_period_label(start, end):
    return f"{start:%Y-%m-%d} → {end:%Y-%m-%d} (end exclusive, UTC)"


,Input,Value
0,Strategy source,/Users/moo/code/strategies/strategy/hyper-ai-v...
1,Strategy SHA-256,5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e...
2,Guard record,/Users/moo/code/strategies/deploy/hyper-ai-v10...
3,Decision cycle,CycleDuration.cycle_2d
4,Initial equity (USD),150000


,Input,Value
0,Parent notebook,01-initial.ipynb
1,Sizing harness SHA-256,d198d05f232e9ecaec600f90db106c758f164e3a72203a...
2,Treatment,Original requested dollars; no redistribution ...
3,Max vaults / concentration / TVL share,6 / 33% / 33%


## Allocation sanity checks

With a $100,000 budget and original 70%/20%/10% shares, the 33% concentration limit lets the parent redistribute to $33,000 in each vault. The treatment preserves the $20,000 and $10,000 original requests for the other two vaults. Also check zero-weight warm-up and the case where no cap binds.

In [2]:
from tradeexecutor.strategy.alpha_model import TradingPairSignal
from tradeexecutor.strategy.fixed_size_risk import FixedSizeRiskModel
from tradeexecutor.state.identifier import AssetIdentifier, TradingPairIdentifier


def allocator_example(mode, weights, max_weight):
    model = AuditedAlphaModel(datetime.datetime(2026, 1, 1), sizing_mode=mode, allocation_ledger=[])
    reserve = AssetIdentifier(1, "0x0000000000000000000000000000000000000001", "USDC", 6)
    for pid, weight in enumerate(weights, start=1):
        base = AssetIdentifier(1, f"0x{pid + 1:040x}", f"V{pid}", 18)
        pair = TradingPairIdentifier(base, reserve, f"0x{pid + 10:040x}", "0x0000000000000000000000000000000000000020", internal_id=pid)
        model.signals[pid] = TradingPairSignal(pair=pair, signal=weight, raw_weight=weight, old_weight=0.0)
    model.normalise_weights(investable_equity=100_000.0, max_weight=max_weight, max_positions=6,
                            size_risk_model=FixedSizeRiskModel(pricing_model=None), waterfall=False)
    return model

example_rows = []
for mode, expected in (("parent", 99_000.0), ("accepted_dollar_cap", 63_000.0)):
    model = allocator_example(mode, [0.7, 0.2, 0.1], 0.33)
    assert abs(model.accepted_investable_equity - expected) < 1e-6
    example_rows.append({"Allocator": mode, "A (USD)": model.signals[1].position_target,
                         "B (USD)": model.signals[2].position_target,
                         "C (USD)": model.signals[3].position_target,
                         "Cash (USD)": 100_000.0 - model.accepted_investable_equity})
    zero = allocator_example(mode, [0.0, 0.0, 0.0], 0.33)
    assert zero.accepted_investable_equity == 0
    uncapped = allocator_example(mode, [0.7, 0.2, 0.1], 1.0)
    assert abs(uncapped.accepted_investable_equity - 100_000.0) < 1e-6
display(pd.DataFrame(example_rows))

,Allocator,A (USD),B (USD),C (USD),Cash (USD)
0,parent,33000.0,33000.0,33000.0,1000.0
1,accepted_dollar_cap,33000.0,20000.0,10000.0,37000.0


## Prepare the universe and indicators

Load v11's full indicator history once, then derive the long window from valid observations for the selected vaults. Both runs use identical indicators and an exclusive end boundary of 5 October 2026, retaining only completed daily candles.

In [3]:
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline

universe_options = UniverseOptions.from_strategy_parameters_class(
    strategy.Parameters, notebook_execution_context
)
universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=universe_options,
    execution_model=None,
)
print("Loading v11 universe and price history; source run used the Trading Strategy vault dataset.")
import sys
sys.path.insert(0, "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
from repaired_data_replay import load_repaired_universe, INDICATOR_DIR, repaired_indicator_storage
strategy_universe = load_repaired_universe(strategy, universe_input)
print(f"Loaded {strategy_universe.get_pair_count()} pairs; calculating indicators with one worker.")
indicator_data = calculate_and_load_indicators_inline(
    indicator_storage_path=INDICATOR_DIR,
    strategy_universe=strategy_universe,
    parameters=parameters,
    create_indicators=strategy.create_indicators,
    execution_context=notebook_execution_context,
    max_workers=1,
)
print(f"Indicator combinations: {len(indicator_data.indicator_combinations)}")

# Exclude benchmark pairs when determining the available vault history.
vault_history = []
for pair in strategy_universe.iterate_pairs():
    if pair.is_vault():
        candles = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)
        valid = candles.loc[candles["close"].gt(0)]
        if len(valid):
            vault_history.append({
                "Pair id": pair.internal_id,
                "First observation": valid.index.min(),
                "Last observation": valid.index.max(),
            })
vault_history_df = pd.DataFrame(vault_history)
long_start = pd.Timestamp(vault_history_df["First observation"].min()).to_pydatetime()
assert long_start < parameters.backtest_start
# Preserve the source's exclusive end boundary: no incomplete 5 October candle.
periods = {
    "Short (source reference)": (parameters.backtest_start, parameters.backtest_end),
    "Long (all available vault history)": (long_start, parameters.backtest_end),
}
display(pd.DataFrame([
    {"Period": label, "Start (UTC)": start, "End (exclusive, UTC)": end,
     "Calendar days": (end - start).days, "Vaults with history": len(vault_history_df)}
    for label, (start, end) in periods.items()
]))

,Period,Start (UTC),"End (exclusive, UTC)",Calendar days,Vaults with history
0,Short (source reference),2026-01-01,2026-10-05,277,514
1,Long (all available vault history),2023-03-01,2026-10-05,1314,514


## Run both strategies over both periods

Each run resets mutable vault fee estimates and starts with fresh portfolio state. Allow a few minutes for the four backtests; progress is reported below. Cash exposure is averaged over all calendar days, including the initial idle period.

In [4]:
runs = {}
jobs = [(period_label, mode, start, end) for period_label, (start, end) in periods.items()
        for mode in ("parent", "accepted_dollar_cap")]
for period_label, mode, start, end in tqdm(jobs, desc="Experiment 02 backtests", total=len(jobs)):
    label = f"{period_label.split()[0]} — {'v11 parent' if mode == 'parent' else 'accepted-dollar cap'}"
    allocation_ledger = []
    run_parameters = deepcopy(parameters)
    run_parameters["backtest_start"] = start
    run_parameters["backtest_end"] = end
    # Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
    strategy.apply_vault_redemption_capital_fee(
        strategy_universe, float(run_parameters.vault_redemption_capital_fee)
    )
    print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
    result = run_backtest_inline(
        engine_version=strategy.trading_strategy_engine_version,
        client=client,
        decide_trades=make_decide_trades(strategy, mode, allocation_ledger),
        universe=strategy_universe,
        indicator_storage=repaired_indicator_storage(strategy_universe),
        indicator_combinations=indicator_data.indicator_combinations,
        parameters=run_parameters,
        cycle_duration=run_parameters.cycle_duration,
        initial_deposit=run_parameters.initial_cash,
        reserve_currency=ReserveCurrency.usdc,
        trade_routing=run_parameters.routing,
        max_slippage=run_parameters.slippage_tolerance_pct,
        start_at=start,
        end_at=end,
        max_workers=1,
        name=f"Experiment 02 — {label}",
        strategy_logging=False,
    )
    state = result.state
    equity = calculate_equity_curve(state)
    # Match the daily-resampled metric convention documented in hyper-ai-v11.py.
    daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
    positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
    cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
    trades = list(state.portfolio.get_all_trades())
    position_counts_by_cycle = {}
    for position_stats in state.stats.positions.values():
        for snapshot in position_stats:
            if snapshot.value > 0:
                position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
    metrics = {
        "Initial equity (USD)": float(run_parameters.initial_cash),
        "Final equity (USD)": float(equity.iloc[-1]),
        "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
        "CAGR": float(calculate_cagr(daily_returns)),
        "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
        "Sortino": float(calculate_sortino(daily_returns, periods=365)),
        "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
        "Decision cycles": len(cycle_stamps),
        "Distinct positions": len(positions),
        "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
        "Trades": len(trades),
    }
    daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
    reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
    daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
    cash_weight = daily_weights[reserve_symbol] / daily_equity
    metrics["Mean cash weight"] = float(cash_weight.mean())
    allocation_df = pd.DataFrame(allocation_ledger)
    if len(allocation_df):
        allocation_df["Avoided refill (USD)"] = (
            allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
        ).clip(lower=0.0)
        if mode == "accepted_dollar_cap":
            assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
            assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
    assert metrics["Maximum vaults held"] <= 6
    runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
                   "daily_returns": daily_returns, "metrics": metrics,
                   "period": period_label, "mode": mode, "allocation_df": allocation_df,
                   "cash_weight": cash_weight,
                   "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
                   "First trade": min((t.opened_at for t in trades), default=None)}


metrics_df = pd.DataFrame({label: run["metrics"] for label, run in runs.items()})
operational_rows = ["Initial equity (USD)", "Final equity (USD)", "Decision cycles",
                    "Distinct positions", "Maximum vaults held", "Trades", "Mean cash weight"]
operational_table = metrics_df.loc[operational_rows].style.format("{:,.2f}")
operational_table.format("{:.2%}", subset=pd.IndexSlice[["Mean cash weight"], :])
display(operational_table)
display(pd.DataFrame([
    {"Run": label, "First decision": run["First decision"], "Last decision": run["Last decision"],
     "First trade": run["First trade"]} for label, run in runs.items()
]))
from repaired_data_replay import RUN_ROOT, save_metrics
save_metrics(runs,RUN_ROOT/"02")


,Short — v11 parent,Short — accepted-dollar cap,Long — v11 parent,Long — accepted-dollar cap
Initial equity (USD),"150,000.00","150,000.00","150,000.00","150,000.00"
Final equity (USD),"194,520.81","160,350.69","254,103.35","161,561.41"
Decision cycles,139.00,139.00,658.00,658.00
Distinct positions,101.00,112.00,214.00,215.00
Maximum vaults held,6.00,6.00,6.00,6.00
Trades,581.00,474.00,"1,074.00",911.00
Mean cash weight,8.83%,51.32%,41.81%,70.57%


,Run,First decision,Last decision,First trade
0,Short — v11 parent,2026-01-01,2026-10-04,2026-01-01
1,Short — accepted-dollar cap,2026-01-01,2026-10-04,2026-01-01
2,Long — v11 parent,2023-02-28,2026-10-04,2023-05-31
3,Long — accepted-dollar cap,2023-02-28,2026-10-04,2023-05-31


## Side-by-side key metrics

Use the existing trade-executor QuantStats table. Calculate each column on its own daily returns before combining the four columns; short-period returns are not padded to the long window.

In [5]:
from tradeexecutor.analysis.advanced_metrics import calculate_advanced_metrics, AdvancedMetricsMode

performance_tables = []
for label, run in runs.items():
    table = calculate_advanced_metrics(
        run["daily_returns"],
        mode=AdvancedMetricsMode.full,
        periods_per_year=365,
        convert_to_daily=False,
        display=True,
    )
    performance_tables.append(table.rename(columns={"Strategy": label}))
key_metrics_comparison = pd.concat(performance_tables, axis=1)
key_metrics_comparison.index.name = "Metric"
display(key_metrics_comparison)

,Short — v11 parent,Short — accepted-dollar cap,Long — v11 parent,Long — accepted-dollar cap
Metric,,,,
Start Period,2026-01-01,2026-01-01,2023-02-28,2023-02-28
End Period,2026-10-04,2026-10-04,2026-10-04,2026-10-04
Risk-Free Rate,0.0%,0.0%,0.0%,0.0%
Time in Market,50.0%,48.0%,31.0%,25.0%
Cumulative Return,29.81%,6.97%,69.4%,7.71%
CAGR﹪,41.2%,9.31%,15.77%,2.08%
Sharpe,1.96,1.31,1.24,0.35
Prob. Sharpe Ratio,96.5%,84.83%,98.89%,72.9%
Smart Sharpe,1.94,1.3,1.23,0.35


## Equity curve comparison

In [6]:
for period_label in periods:
    frames = []
    for label, run in runs.items():
        if run["period"] == period_label:
            frame = run["equity"].rename("Portfolio equity (USD)").rename_axis("Date").reset_index()
            frame["Strategy"] = "v11 parent" if run["mode"] == "parent" else "Accepted-dollar cap"
            frames.append(frame)
    fig = px.line(pd.concat(frames), x="Date", y="Portfolio equity (USD)", color="Strategy",
                  title=f"Experiment 02 — {period_label}")
    with_backtest_period(fig, backtest_period_label(*periods[period_label]))
    fig.show(width=1500, height=700)

## Quarterly profit heatmaps

Use the existing trade-executor returns heatmap with calendar-quarter aggregation. Each cell shows compounded portfolio profit/loss in percent, including changes in the value of open positions. The calculation uses the same daily returns as the metrics table.

Boundary quarters contain only observed dates: long-period 2023 Q1 starts with the cash-only 28 February cycle, and 2026 Q4 covers 1–4 October in both runs. These are partial quarters. Blank cells are outside the run's window; 0% means the observed portfolio return was zero.

In [7]:
from matplotlib import pyplot as plt
from tradeexecutor.visual.equity_curve import visualise_returns_over_time

for label, run in runs.items():
    fig = visualise_returns_over_time(run["daily_returns"], period="quarter")
    fig.axes[0].set_title(f"Quarterly portfolio profit/loss — {label}" + "\nBacktest: " + backtest_period_label(run["parameters"].backtest_start, run["parameters"].backtest_end))
    display(fig)
    plt.close(fig)

## Absolute asset allocation

Each stacked chart shows the USD value of every vault holding and reserve cash. Values are absolute, rather than normalised to 100%.

In [8]:
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset

for label, run in runs.items():
    result = run["result"]
    run_parameters = run["parameters"]
    charts = strategy.create_charts(
        timestamp=None,
        parameters=run_parameters,
        strategy_universe=result.strategy_universe,
        execution_context=notebook_execution_context,
    )
    chart_renderer = ChartBacktestRenderingSetup(
        registry=charts,
        strategy_input_indicators=result.indicators,
        state=result.state,
        backtest_start_at=run_parameters.backtest_start,
        backtest_end_at=run_parameters.backtest_end,
    )
    asset_weights_fig = chart_renderer.render(equity_curve_by_asset)
    asset_weights_fig.update_layout(title=f"Absolute asset allocation (USD) — {label}", width=1500, height=900)
    # Keep every vault in the static legend visible across the longer history.
    chart_height = max(900, 170 + 20 * len(asset_weights_fig.data))
    with_backtest_period(asset_weights_fig, backtest_period_label(run_parameters.backtest_start, run_parameters.backtest_end))
    asset_weights_fig.show(width=1500, height=chart_height)

## Parent replication and allocation audit

Verify both parent controls against the saved experiment 01 results. The same-state counterfactual measures the dollars the parent would redistribute using each treatment run's own pre-trade state. The ledger also checks that post-sizing operational checks only reduce the capped targets. Caps constrain desired targets; lock-ups, fees, rounding and minimum-trade gates may cause executed holdings to differ.

In [9]:
references = {
    "Short (source reference)": {"Final equity (USD)": 194_929.0, "Cumulative return": 0.3008,
        "CAGR": 0.4159, "Sharpe": 2.00, "Sortino": 3.56, "Maximum drawdown": -0.0640,
        "Decision cycles": 139, "Distinct positions": 101, "Maximum vaults held": 6, "Trades": 582},
    "Long (all available vault history)": {"Final equity (USD)": 254_510.70, "Cumulative return": 0.6967,
        "CAGR": 0.1582, "Sharpe": 1.25, "Sortino": 1.91, "Maximum drawdown": -0.0792,
        "Decision cycles": 658, "Distinct positions": 214, "Maximum vaults held": 6, "Trades": 1074},
}
tolerances = {"Final equity (USD)": 2.0, "Cumulative return": 0.00015, "CAGR": 0.00015,
              "Sharpe": 0.015, "Sortino": 0.015, "Maximum drawdown": 0.00015,
              "Decision cycles": 0, "Distinct positions": 0, "Maximum vaults held": 0, "Trades": 0}
rows = []
for label, run in runs.items():
    if run["mode"] != "parent":
        continue
    for metric, expected in references[run["period"]].items():
        actual = run["metrics"][metric]
        rows.append({"Run": label, "Metric": metric, "Reference": expected, "Rerun": actual,
                     "Match": abs(actual - expected) <= tolerances[metric]})
replication_df = pd.DataFrame(rows)
display(replication_df)
# Historical benchmark comparison; repaired-input parent controls are checked below.

allocation_summaries = []
for label, run in runs.items():
    ledger = run["allocation_df"]
    per_cycle = ledger.groupby("Date").agg(
        budget=("Deployable budget (USD)", "first"),
        parent_accepted=("Parent accepted, same state (USD)", "sum"),
        accepted=("Accepted allocation (USD)", "sum"),
        final_target=("Final target (USD)", "sum"),
        avoided_refill=("Avoided refill (USD)", "sum"),
    )
    run["allocation_summary"] = per_cycle
    allocation_summaries.append({"Run": label, "Audited sizing cycles": len(per_cycle),
        "Cycles with changed targets": int(per_cycle.avoided_refill.gt(0.01).sum()),
        "Mean avoided refill per sized cycle (USD)": per_cycle.avoided_refill.mean(),
        "Mean final target / budget": (per_cycle.final_target / per_cycle.budget.replace(0, float("nan"))).mean()})
    if run["mode"] == "accepted_dollar_cap":
        assert per_cycle.avoided_refill.gt(0.01).any(), "Dollar cap was a no-op"
display(pd.DataFrame(allocation_summaries))

example_run = next(run for run in runs.values() if run["mode"] == "accepted_dollar_cap")
example_date = example_run["allocation_summary"].avoided_refill.idxmax()
example_ledger = example_run["allocation_df"]
display(example_ledger.loc[example_ledger["Date"].eq(example_date), [
    "Date", "Vault", "Original weight", "Intended allocation (USD)",
    "Parent accepted, same state (USD)", "Accepted allocation (USD)", "Final target (USD)"
]])
from repaired_data_replay import RUN_ROOT, save_metrics
import json
save_metrics(runs,RUN_ROOT/"02")
fresh_parent=json.loads((RUN_ROOT/"01/metrics.json").read_text())
for run in runs.values():
    if run["mode"]=="parent":
        for metric in ["Final equity (USD)","CAGR","Sharpe","Maximum drawdown","Trades"]:
            assert abs(run["metrics"][metric]-fresh_parent[run["period"]][metric])<1e-6, (run["period"],metric)


,Run,Metric,Reference,Rerun,Match
0,Short — v11 parent,Final equity (USD),194929.0000,194520.812686,False
1,Short — v11 parent,Cumulative return,0.3008,0.298078,False
2,Short — v11 parent,CAGR,0.4159,0.412004,False
3,Short — v11 parent,Sharpe,2.0000,1.959590,False
4,Short — v11 parent,Sortino,3.5600,3.429825,False
5,Short — v11 parent,Maximum drawdown,-0.0640,-0.064031,True
6,Short — v11 parent,Decision cycles,139.0000,139.000000,True
7,Short — v11 parent,Distinct positions,101.0000,101.000000,True
8,Short — v11 parent,Maximum vaults held,6.0000,6.000000,True
9,Short — v11 parent,Trades,582.0000,581.000000,False


,Run,Audited sizing cycles,Cycles with changed targets,Mean avoided refill per sized cycle (USD),Mean final target / budget
0,Short — v11 parent,139,0,0.000000,0.925352
1,Short — accepted-dollar cap,139,139,67848.366233,0.492929
2,Long — v11 parent,612,0,0.000000,0.636289
3,Long — accepted-dollar cap,612,511,48373.756183,0.321211


,Date,Vault,Original weight,Intended allocation (USD),"Parent accepted, same state (USD)",Accepted allocation (USD),Final target (USD)
341,2026-04-15,AJ Pro,0.999946,153743.835550,4348.835111,4348.835111,4348.835111
342,2026-04-15,pmalt,0.000026,3.951997,50738.202335,3.951997,3.951997
343,2026-04-15,Realist Capital,0.000014,2.175043,49438.750624,2.175043,2.175043
344,2026-04-15,Citadel,0.000007,1.078851,24522.304966,1.078851,1.078851
345,2026-04-15,Delta_01,0.000004,0.590672,13425.982613,0.590672,0.590672
346,2026-04-15,Hyperrr,0.000003,0.496174,11278.052639,0.496174,0.496174
347,2026-04-15,[ Systemic Strategies ] L/S Grids,0.000000,0.000000,0.000000,0.000000,0.000000


## Robustness analysis

Reconstruct each position's PnL from successful USDC trade flows and marked holdings, including final realised profit for closed positions. Reconcile the sum to equity minus the initial deposit, then group by vault address. Show the top vaults and positions and attribute the three strongest portfolio days.

Use the repository's `curator.py` classifications and quarantine windows to annotate influential vaults. Binance BTC and ETH opening prices provide broad-market context over the matching prior-snapshot-to-current-snapshot UTC interval. With two-day decisions, a return booked on one date can cover more than one calendar day; compare that interval rather than a single market day. The best-day-zero and leading-vault-PnL removal figures are fixed-path sensitivities, not executable alternative strategies; they do not establish out-of-sample performance. Cash exposure and the two overlapping windows cannot by themselves separate reduced market exposure from better sizing.

In [10]:
from tradingstrategy.binance.price import fetch_binance_price
from tradeexecutor.curator.curator import EXCLUDED_VAULTS, get_hyperliquid_concentration_tweak, is_quarantined

# Incrementally cached Binance reference series, used only for best-day context.
market_open_prices = {}
for symbol in ("BTCUSDT", "ETHUSDT"):
    prices = fetch_binance_price(symbol=symbol, start=long_start - datetime.timedelta(days=7))
    market_open_prices[symbol] = prices["open"]

curated_quality = get_hyperliquid_concentration_tweak()
robustness_rows = []
contributor_tables = []
jump_tables = []
for label, run in runs.items():
    state = run["result"].state
    daily = run["daily_returns"]
    daily_equity = run["equity"].resample("1D").last().reindex(daily.index).ffill()
    curves, position_metadata = calculate_position_profit_curves(state, daily.index)
    portfolio_pnl = daily_equity - float(run["parameters"].initial_cash)
    residual = curves.sum(axis=1) - portfolio_pnl
    assert residual.abs().max() < 0.01, f"Position attribution does not reconcile: {label}, {residual.abs().max()}"
    vault_curves = {}
    for address, group in position_metadata.groupby("Address"):
        vault_curves[address] = curves[group["Position id"].tolist()].sum(axis=1)
    vault_curves = pd.DataFrame(vault_curves)
    final_profits = vault_curves.iloc[-1].sort_values(ascending=False)
    leader = final_profits.index[0]
    leader_profit = float(final_profits.iloc[0])
    positive_profit_total = final_profits.clip(lower=0.0).sum()
    best_day = daily.idxmax()
    without_best_day = daily.copy()
    without_best_day.loc[best_day] = 0.0
    # Fixed-ledger attribution sensitivity: retain the original trades and sizing.
    without_leader_equity = daily_equity - vault_curves[leader]
    without_leader_returns = calculate_returns(without_leader_equity)
    robustness_rows.append({
        "Run": label, "Best day": best_day, "Best day return": float(daily.loc[best_day]),
        "Return kurtosis (excess)": float(daily.kurt()),
        "CAGR with best day set to zero": float(calculate_cagr(without_best_day)),
        "Sharpe with best day set to zero": float(calculate_sharpe(without_best_day, periods=365)),
        "Leading vault": position_metadata.loc[position_metadata.Address.eq(leader), "Vault"].iloc[0],
        "Leading vault profit (USD)": leader_profit,
        "Leading vault share of positive vault profits": float(leader_profit / positive_profit_total),
        "CAGR without leader PnL (fixed ledger)": float(calculate_cagr(without_leader_returns)),
        "Sharpe without leader PnL (fixed ledger)": float(calculate_sharpe(without_leader_returns, periods=365)),
        "Max attribution residual (USD)": float(residual.abs().max()),
    })
    names = position_metadata.groupby("Address").Vault.first()
    contribution = pd.DataFrame({"Address": final_profits.index, "Profit (USD)": final_profits.values})
    contribution["Vault"] = contribution.Address.map(names)
    contribution["Run"] = label
    contribution["Curated quality"] = contribution.Address.map(lambda address: getattr(curated_quality.get(address), "value", "unrated"))
    contribution["Curator excluded"] = contribution.Address.isin(EXCLUDED_VAULTS)
    contributor_tables.append(contribution.head(5))
    position_metadata["Run"] = label
    run["position_metadata"] = position_metadata
    run["vault_profit_curves"] = vault_curves
    daily_pnl = vault_curves.diff().fillna(vault_curves.iloc[0])
    top_days = daily.nlargest(3).index
    for day in top_days:
        gains = daily_pnl.loc[day].sort_values(ascending=False)
        positive = gains.clip(lower=0.0).sum()
        top_address = gains.index[0]
        snapshot_dates = pd.DatetimeIndex(run["equity"].index).normalize().unique()
        previous_snapshot = snapshot_dates[snapshot_dates < day][-1]
        previous_equity = float(daily_equity.loc[previous_snapshot])
        market_interval_returns = {
            symbol: float(series.loc[day] / series.loc[previous_snapshot] - 1.0)
            for symbol, series in market_open_prices.items()
        }
        jump_tables.append({
            "Run": label, "Date": day, "Portfolio return": float(daily.loc[day]),
            "Largest contributing vault": names[top_address],
            "Vault PnL that day (USD)": float(gains.iloc[0]),
            "Vault contribution / prior equity": float(gains.iloc[0] / previous_equity),
            "Share of positive vault PnL": float(gains.iloc[0] / positive) if positive else 0.0,
            "Previous snapshot": previous_snapshot,
            "BTC over snapshot interval": market_interval_returns["BTCUSDT"],
            "ETH over snapshot interval": market_interval_returns["ETHUSDT"],
            "Vault quarantined that day": is_quarantined(top_address, day.to_pydatetime()),
        })
robustness_df = pd.DataFrame(robustness_rows)
contribution_df = pd.concat(contributor_tables, ignore_index=True)
jumps_df = pd.DataFrame(jump_tables)
display(robustness_df)
display(contribution_df)
display(jumps_df)

position_tables = []
for label, run in runs.items():
    table = run["position_metadata"].sort_values("Final profit (USD)", ascending=False).head(3)
    position_tables.append(table)
display(pd.concat(position_tables, ignore_index=True))

,Run,Best day,Best day return,Return kurtosis (excess),CAGR with best day set to zero,Sharpe with best day set to zero,Leading vault,Leading vault profit (USD),Leading vault share of positive vault profits,CAGR without leader PnL (fixed ledger),Sharpe without leader PnL (fixed ledger),Max attribution residual (USD)
0,Short — v11 parent,2026-06-26,0.068871,13.670515,0.292955,1.620008,Realist Capital,24196.229340,0.447703,0.184526,1.097709,3.637979e-11
1,Short — accepted-dollar cap,2026-02-28,0.018699,39.344446,0.066675,1.003495,pmalt,4257.495237,0.293620,0.054915,0.822736,2.910383e-11
2,Long — v11 parent,2025-08-14,0.054258,39.147227,0.140813,1.147279,Realist Capital,25183.629423,0.172916,0.124599,1.048062,7.275958e-11
3,Long — accepted-dollar cap,2025-10-15,0.033401,263.870660,0.011565,0.218205,22Cap,6351.321500,0.193729,0.009530,0.181295,3.274181e-11


,Address,Profit (USD),Vault,Run,Curated quality,Curator excluded
0,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,24196.229340,Realist Capital,Short — v11 parent,unrated,False
1,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,10689.272691,DOEZOE,Short — v11 parent,unrated,False
2,0x4dec0a851849056e259128464ef28ce78afa27f6,6623.691469,pmalt,Short — v11 parent,high,False
3,0xba939edf38c0ae0cc689c98b492e0535f43e4550,3624.343404,22Cap,Short — v11 parent,unrated,False
4,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2286.660570,Citadel,Short — v11 parent,unrated,False
5,0x4dec0a851849056e259128464ef28ce78afa27f6,4257.495237,pmalt,Short — accepted-dollar cap,high,False
6,0xba939edf38c0ae0cc689c98b492e0535f43e4550,2967.269752,22Cap,Short — accepted-dollar cap,unrated,False
7,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,1561.554794,Realist Capital,Short — accepted-dollar cap,unrated,False
8,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,1428.979487,Citadel,Short — accepted-dollar cap,unrated,False
9,0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,1402.659287,Goon Edging,Short — accepted-dollar cap,unrated,False


,Run,Date,Portfolio return,Largest contributing vault,Vault PnL that day (USD),Vault contribution / prior equity,Share of positive vault PnL,Previous snapshot,BTC over snapshot interval,ETH over snapshot interval,Vault quarantined that day
0,Short — v11 parent,2026-06-26,0.068871,Realist Capital,6929.951861,0.041740,0.583981,2026-06-24,-0.046863,-0.059545,False
1,Short — v11 parent,2026-06-24,0.041142,Realist Capital,8968.757978,0.056242,1.000000,2026-06-22,-0.009120,-0.023322,False
2,Short — v11 parent,2026-07-08,0.036604,Realist Capital,6207.538547,0.033220,0.870987,2026-07-06,-0.004493,-0.007952,False
3,Short — accepted-dollar cap,2026-02-28,0.018699,pmalt,1965.465770,0.012819,0.685539,2026-02-26,-0.031122,-0.062037,False
4,Short — accepted-dollar cap,2026-06-26,0.014795,Goon Edging,1983.607541,0.012824,0.738046,2026-06-24,-0.046863,-0.059545,False
5,Short — accepted-dollar cap,2026-08-19,0.014300,Citadel,1915.486236,0.012101,0.812906,2026-08-17,0.029021,0.022303,False
6,Long — v11 parent,2025-08-14,0.054258,DailyTradeAI Low risk,3611.727339,0.019293,0.352984,2025-08-12,0.038930,0.124571,False
7,Long — v11 parent,2026-06-26,0.053091,Realist Capital,7430.079007,0.033172,0.602855,2026-06-24,-0.046863,-0.059545,False
8,Long — v11 parent,2024-12-25,0.049095,Delta_01,6489.446688,0.038161,0.777292,2024-12-23,0.036532,0.064397,False
9,Long — accepted-dollar cap,2025-10-15,0.033401,Growi HF,3299.660991,0.021854,0.607806,2025-10-13,-0.016795,-0.006570,False


,Position id,Vault,Address,Opened,Closed,Final profit (USD),Run
0,64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,23284.926869,Short — v11 parent
1,98,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,2026-09-18,NaT,11815.190940,Short — v11 parent
2,7,pmalt,0x4dec0a851849056e259128464ef28ce78afa27f6,2026-01-05,2026-05-21,8291.949737,Short — v11 parent
3,88,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2026-08-05,2026-08-27,3012.417960,Short — accepted-dollar cap
4,19,pmalt,0x4dec0a851849056e259128464ef28ce78afa27f6,2026-01-25,2026-04-15,2424.056448,Short — accepted-dollar cap
5,92,AceVault Hyper01,0x9e02aca9865e1859bb7865f6f64801e804a173df,2026-08-17,2026-08-29,2136.295287,Short — accepted-dollar cap
6,177,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,24149.677338,Long — v11 parent
7,25,Delta_01,0x3005fade4c0df5e1cd187d7062da359416f0eb8e,2024-12-01,2025-02-07,18605.941433,Long — v11 parent
8,2,Hyperliquidity Provider (HLP),0xdfc24b077bc1425ad1dea75bcb6f8158e10df303,2023-08-09,2024-05-23,17763.479053,Long — v11 parent
9,2,Hyperliquidity Provider (HLP),0xdfc24b077bc1425ad1dea75bcb6f8158e10df303,2023-08-09,2023-11-09,3720.744083,Long — accepted-dollar cap


## Why did the cap underperform?

The original accepted-dollar proposal also requests a causal cash-matched parent. The two additional diagnostic runs below use the parent's accepted allocation proportions, scaled down to the total accepted by the original-request cap **on each control's own pre-trade state**. They retain the same v11 ranking, caps, fees and trade thresholds. This matches decision-time sizing budgets, not the eventual cash path: different holdings, prices, operational restrictions and hysteresis can cause realised exposure to diverge.

Trade attribution then compares all parent and cap vaults, including losers, turnover and modelled redemption costs. The audit distinguishes desired dollar targets from actual execution: the $5 absolute deposit minimum, equity-scaled buy/sell thresholds and 0.5% accepted-basket cleanup can suppress small positions. These shared thresholds are not silently removed to improve the treatment.

Parameter sensitivities below are **same-ledger sizing calculations only**, using the treatment's observed selected names and budget. They show how concentration, TVL limits and raw inverse-variance weights constrain deployment; they are not backtested alternative strategies or return forecasts.


In [11]:
control_runs = {}
jobs = [(period_label, mode, start, end) for period_label, (start, end) in periods.items()
        for mode in ("cash_matched_parent",)]
for period_label, mode, start, end in tqdm(jobs, desc="Cash-matched controls", total=len(jobs)):
    label = f"{period_label.split()[0]} — cash-matched parent"
    allocation_ledger = []
    run_parameters = deepcopy(parameters)
    run_parameters["backtest_start"] = start
    run_parameters["backtest_end"] = end
    # Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
    strategy.apply_vault_redemption_capital_fee(
        strategy_universe, float(run_parameters.vault_redemption_capital_fee)
    )
    print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
    result = run_backtest_inline(
        engine_version=strategy.trading_strategy_engine_version,
        client=client,
        decide_trades=make_decide_trades(strategy, mode, allocation_ledger),
        universe=strategy_universe,
        indicator_storage=repaired_indicator_storage(strategy_universe),
        indicator_combinations=indicator_data.indicator_combinations,
        parameters=run_parameters,
        cycle_duration=run_parameters.cycle_duration,
        initial_deposit=run_parameters.initial_cash,
        reserve_currency=ReserveCurrency.usdc,
        trade_routing=run_parameters.routing,
        max_slippage=run_parameters.slippage_tolerance_pct,
        start_at=start,
        end_at=end,
        max_workers=1,
        name=f"Experiment 02 — {label}",
        strategy_logging=False,
    )
    state = result.state
    equity = calculate_equity_curve(state)
    # Match the daily-resampled metric convention documented in hyper-ai-v11.py.
    daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
    positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
    cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
    trades = list(state.portfolio.get_all_trades())
    position_counts_by_cycle = {}
    for position_stats in state.stats.positions.values():
        for snapshot in position_stats:
            if snapshot.value > 0:
                position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
    metrics = {
        "Initial equity (USD)": float(run_parameters.initial_cash),
        "Final equity (USD)": float(equity.iloc[-1]),
        "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
        "CAGR": float(calculate_cagr(daily_returns)),
        "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
        "Sortino": float(calculate_sortino(daily_returns, periods=365)),
        "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
        "Decision cycles": len(cycle_stamps),
        "Distinct positions": len(positions),
        "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
        "Trades": len(trades),
    }
    daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
    reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
    daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
    cash_weight = daily_weights[reserve_symbol] / daily_equity
    metrics["Mean cash weight"] = float(cash_weight.mean())
    allocation_df = pd.DataFrame(allocation_ledger)
    if len(allocation_df):
        allocation_df["Avoided refill (USD)"] = (
            allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
        ).clip(lower=0.0)
        if mode == "accepted_dollar_cap":
            assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
            assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
    assert metrics["Maximum vaults held"] <= 6
    control_runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
                   "daily_returns": daily_returns, "metrics": metrics,
                   "period": period_label, "mode": mode, "allocation_df": allocation_df,
                   "cash_weight": cash_weight,
                   "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
                   "First trade": min((t.opened_at for t in trades), default=None)}


comparison_runs = {**runs, **control_runs}
display(pd.DataFrame({label: run["metrics"] for label, run in comparison_runs.items()}).loc[
    ["Final equity (USD)", "CAGR", "Sharpe", "Maximum drawdown", "Mean cash weight", "Trades"]
].style.format("{:,.4f}"))


,Short — v11 parent,Short — accepted-dollar cap,Long — v11 parent,Long — accepted-dollar cap,Short — cash-matched parent,Long — cash-matched parent
Final equity (USD),"194,520.8127","160,350.6852","254,103.3482","161,561.4129","171,453.0330","172,334.2611"
CAGR,0.4120,0.0931,0.1577,0.0208,0.1943,0.0393
Sharpe,1.9596,1.3095,1.2381,0.3546,1.9214,0.5967
Maximum drawdown,-0.0640,-0.0422,-0.0792,-0.0792,-0.0338,-0.0792
Mean cash weight,0.0883,0.5132,0.4181,0.7057,0.5014,0.7132
Trades,581.0000,474.0000,"1,074.0000",911.0000,612.0000,"1,297.0000"


In [12]:
import numpy as np

ARTIFACT_DIR = RESEARCH_DIR / "_artifacts-restored-2026-10-07/02"
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
trade_frames = []
for label, run in comparison_runs.items():
    state = run["result"].state
    rows = []
    for position in state.portfolio.get_all_positions():
        for trade in position.get_successful_trades():
            if not (trade.is_buy() or trade.is_sell()):
                continue
            fee_rate = float(trade.other_data.get("backtest_vault_redemption_fee", 0.0))
            gross_sell = abs(float(trade.executed_quantity or 0.0)) * float(trade.planned_mid_price or 0.0) if trade.is_sell() else 0.0
            rows.append({"Run": label, "Date": pd.Timestamp(trade.executed_at),
                "Address": str(position.pair.pool_address).lower(),
                "Vault": position.pair.get_vault_name(), "Position id": position.position_id,
                "Side": "buy" if trade.is_buy() else "sell",
                "Reserve (USD)": float(trade.executed_reserve or 0.0),
                "Modelled redemption cost (USD)": gross_sell * fee_rate,
                "Modelled capital friction (USD)": gross_sell * float(run["parameters"].vault_redemption_capital_fee),
                "Notes": trade.notes or ""})
    run["trades_df"] = pd.DataFrame(rows)
    trade_frames.append(run["trades_df"])
    run["allocation_df"].to_csv(ARTIFACT_DIR / f"{label.split()[0].lower()}-{run['mode']}-allocations.csv", index=False)
    state.write_json_file(ARTIFACT_DIR / f"{label.split()[0].lower()}-{run['mode']}-state.json")
trades_df = pd.concat(trade_frames, ignore_index=True)
trades_df.to_csv(ARTIFACT_DIR / "trades.csv", index=False)

execution_rows = []
for label, run in comparison_runs.items():
    t = run["trades_df"]
    ledger = run["allocation_df"]
    positive = ledger["Final target (USD)"].gt(0)
    tiny = positive & ledger["Normalised accepted weight"].lt(float(parameters.min_portfolio_weight_pct))
    below_buy = positive & ledger["Final target (USD)"].lt(ledger["Deployable budget (USD)"] / float(parameters.allocation_pct) * float(parameters.individual_rebalance_min_threshold_of_initial_cash_pct))
    execution_rows.append({"Run": label,
        "Buys": int(t.Side.eq("buy").sum()), "Sells": int(t.Side.eq("sell").sum()),
        "Buy turnover (USD)": t.loc[t.Side.eq("buy"), "Reserve (USD)"].sum(),
        "Sell proceeds (USD)": t.loc[t.Side.eq("sell"), "Reserve (USD)"].sum(),
        "Modelled redemption costs (USD)": t["Modelled redemption cost (USD)"].sum(),
        "Capital friction component (USD)": t["Modelled capital friction (USD)"].sum(),
        "Positive target rows": int(positive.sum()),
        "Targets below 0.5% basket cleanup": int(tiny.sum()),
        "Targets below approximate buy minimum": int(below_buy.sum()),
        "Positive targets with no planned buy or sell": int((positive & ledger["Planned buy (USD)"].eq(0) & ledger["Planned sells"].eq(0)).sum()),
        "Cleanup full closes": int(t.Notes.str.contains("signal weight is below", regex=False).sum()),
    })
execution_df = pd.DataFrame(execution_rows)
display(execution_df)
execution_df.to_csv(ARTIFACT_DIR / "execution-summary.csv", index=False)

# Same-ledger clipping sensitivities: no return or selection extrapolation.
sizing_rows = []
for label, run in runs.items():
    if run["mode"] != "accepted_dollar_cap":
        continue
    l = run["allocation_df"]
    for date, g in l.groupby("Date"):
        g = g.loc[g["Raw weight"].gt(0)].copy()
        if not len(g):
            continue
        budget = float(g["Deployable budget (USD)"].iloc[0])
        if budget <= 0:
            continue
        raw = g["Raw weight"]
        variants = {
            "v11 inverse variance; 33% / 33%": (raw, .33, .33),
            "same weights; 50% concentration": (raw, .50, .33),
            "same weights; no concentration clip": (raw, 1.0, .33),
            "same weights; no TVL clip": (raw, .33, float("inf")),
            "inverse volatility; 33% / 33%": (raw.pow(.5), .33, .33),
            "equal weights; 33% / 33%": (pd.Series(1.0, index=g.index), .33, .33),
            "inverse variance; daily vol floor 0.1%": (raw.clip(upper=1/.001**2), .33, .33),
        }
        for name, (weights, concentration, tvl_fraction) in variants.items():
            intended = weights / weights.sum() * budget
            accepted = intended.clip(upper=concentration*budget)
            if np.isfinite(tvl_fraction):
                accepted = pd.concat([accepted, g["TVL (USD)"] * tvl_fraction], axis=1).min(axis=1)
            sizing_rows.append({"Run": label, "Date": date, "Sizing calculation": name,
                "Accepted / budget": accepted.sum()/budget,
                "Largest original share": (weights/weights.sum()).max(),
                "Original share >90%": bool((weights/weights.sum()).max() > .90)})
sizing_df = pd.DataFrame(sizing_rows)
display(sizing_df.groupby(["Run", "Sizing calculation"]).agg(
    mean_accepted_fraction=("Accepted / budget", "mean"),
    median_accepted_fraction=("Accepted / budget", "median"),
    concentrated_cycle_fraction=("Original share >90%", "mean")))
sizing_df.to_csv(ARTIFACT_DIR / "sizing-sensitivities.csv", index=False)

trade_attribution = []
for period in ("Short", "Long"):
    parent = runs[f"{period} — v11 parent"]
    cap = runs[f"{period} — accepted-dollar cap"]
    table = pd.DataFrame({"Parent PnL (USD)": parent["vault_profit_curves"].iloc[-1],
                          "Cap PnL (USD)": cap["vault_profit_curves"].iloc[-1]}).fillna(0.0)
    names = pd.concat([parent["position_metadata"], cap["position_metadata"]]).groupby("Address").Vault.first()
    table["Vault"] = table.index.map(names)
    table["Parent minus cap PnL (USD)"] = table["Parent PnL (USD)"]-table["Cap PnL (USD)"]
    for mode_label, run in [("Parent", parent), ("Cap", cap)]:
        buys = run["trades_df"].query("Side == 'buy'").groupby("Address")["Reserve (USD)"].sum()
        table[f"{mode_label} buy turnover (USD)"] = buys.reindex(table.index, fill_value=0.0)
    table["Period"] = period
    assert abs(table["Parent minus cap PnL (USD)"].sum() - (parent["equity"].iloc[-1]-cap["equity"].iloc[-1])) < .01
    trade_attribution.append(table.reset_index(names="Address"))
    display(table.sort_values("Parent minus cap PnL (USD)", ascending=False).head(10))
    display(table.sort_values("Parent minus cap PnL (USD)").head(5))
attribution_df = pd.concat(trade_attribution, ignore_index=True)
attribution_df.to_csv(ARTIFACT_DIR / "parent-cap-attribution.csv", index=False)

# Inspect the short-period winner's actual dated deposits, trims and closures.
realist = attribution_df.query("Period == 'Short' and Vault == 'Realist Capital'").Address.iloc[0]
display(trades_df.loc[trades_df.Address.eq(realist) & trades_df.Run.str.startswith("Short"),
    ["Run", "Date", "Side", "Reserve (USD)", "Modelled redemption cost (USD)", "Position id"]])

# AJ Pro's underlying 90-day return sample preceding the audited decision.
example = runs["Short — accepted-dollar cap"]["allocation_df"]
aj = example.loc[example.Date.eq(pd.Timestamp("2026-04-15")) & example.Vault.eq("AJ Pro")].iloc[0]
pair = strategy_universe.get_pair_by_id(int(aj["Pair id"]))
close = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)["close"]
sample_returns = close.pct_change().loc[close.index < pd.Timestamp("2026-04-15")].tail(int(parameters.inverse_vol_window))
display(pd.DataFrame([{"Vault": "AJ Pro", "Decision": pd.Timestamp("2026-04-15"),
    "Window observations": len(sample_returns), "Exactly flat returns": int(sample_returns.eq(0).sum()),
    "Measured daily standard deviation": sample_returns.std(), "Source daily volatility floor": strategy.VOL_FLOOR,
    "Daily sigma inferred from raw weight": 1 / np.sqrt(float(aj["Raw weight"])),
    "Original requested dollars": aj["Intended allocation (USD)"],
    "Accepted dollars": aj["Accepted allocation (USD)"], "TVL dollars": aj["TVL (USD)"]}]))


,Run,Buys,Sells,Buy turnover (USD),Sell proceeds (USD),Modelled redemption costs (USD),Capital friction component (USD),Positive target rows,Targets below 0.5% basket cleanup,Targets below approximate buy minimum,Positive targets with no planned buy or sell,Cleanup full closes
0,Short — v11 parent,336,245,3.522982e+06,3.501846e+06,13078.320667,3514.924211,834,0,0,348,95
1,Short — accepted-dollar cap,288,186,2.398893e+06,2.364423e+06,5950.861878,2370.374339,834,60,41,449,94
2,Long — v11 parent,614,460,7.211662e+06,7.249735e+06,32359.748818,7282.094262,2846,0,0,1980,208
3,Long — accepted-dollar cap,535,376,3.847001e+06,3.813705e+06,10790.109481,3824.494711,2738,597,556,2003,184
4,Short — cash-matched parent,353,259,2.365587e+06,2.367038e+06,7883.028904,2374.921430,834,8,0,317,95
5,Long — cash-matched parent,767,530,3.822058e+06,3.824374e+06,13050.637243,3837.424348,2846,8,14,1756,207


mean_accepted_fraction  \
Run                         Sizing calculation                                               
Long — accepted-dollar cap  equal weights; 33% / 33%                              0.553858   
                            inverse variance; daily vol floor 0.1%                0.325731   
                            inverse volatility; 33% / 33%                         0.414472   
                            same weights; 50% concentration                       0.381030   
                            same weights; no TVL clip                             0.538095   
                            same weights; no concentration clip                   0.471340   
                            v11 inverse variance; 33% / 33%                       0.322126   
Short — accepted-dollar cap equal weights; 33% / 33%                              0.783477   
                            inverse variance; daily vol floor 0.1%                0.499099   
                            inverse volatility; 33% / 33%                         0.664756   
                            same weights; 50% concentration                       0.557586   
                            same weights; no TVL clip                             0.696730   
                            same weights; no concentration clip                   0.614684   
                            v11 inverse variance; 33% / 33%                       0.496970   

                                                                    median_accepted_fraction  \
Run                         Sizing calculation                                                 
Long — accepted-dollar cap  equal weights; 33% / 33%                                0.478201   
                            inverse variance; daily vol floor 0.1%                  0.330000   
                            inverse volatility; 33% / 33%                           0.397308   
                            same weights; 50% concentration                         0.500000   
                            same weights; no TVL clip                               0.492104   
                            same weights; no concentration clip                     0.504409   
                            v11 inverse variance; 33% / 33%                         0.330000   
Short — accepted-dollar cap equal weights; 33% / 33%                                0.841420   
                            inverse variance; daily vol floor 0.1%                  0.533584   
                            inverse volatility; 33% / 33%                           0.712788   
                            same weights; 50% concentration                         0.555449   
                            same weights; no TVL clip                               0.706806   
                            same weights; no concentration clip                     0.569187   
                            v11 inverse variance; 33% / 33%                         0.533584   

                                                                    concentrated_cycle_fraction  
Run                         Sizing calculation                                                   
Long — accepted-dollar cap  equal weights; 33% / 33%                                   0.068627  
                            inverse variance; daily vol floor 0.1%                     0.418301  
                            inverse volatility; 33% / 33%                              0.370915  
                            same weights; 50% concentration                            0.418301  
                            same weights; no TVL clip                                  0.418301  
                            same weights; no concentration clip                        0.418301  
                            v11 inverse variance; 33% / 33%                            0.418301  
Short — accepted-dollar cap equal weights; 33% / 33%                                   0.000000  
                            inverse variance; 

,Parent PnL (USD),Cap PnL (USD),Vault,Parent minus cap PnL (USD),Parent buy turnover (USD),Cap buy turnover (USD),Period
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,24196.229340,1561.554794,Realist Capital,22634.674546,255713.775845,83548.660538,Short
0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,10689.272691,-7.556763,DOEZOE,10696.829454,42710.481759,945.099200,Short
0x4dec0a851849056e259128464ef28ce78afa27f6,6623.691469,4257.495237,pmalt,2366.196232,247962.092275,159263.659864,Short
0x9e02aca9865e1859bb7865f6f64801e804a173df,1364.471681,428.562462,AceVault Hyper01,935.909219,195396.932811,99610.822757,Short
0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2286.660570,1428.979487,Citadel,857.681083,488255.852772,431925.987192,Short
0xa844d7ac9fa3424c4fd38a25baa23e460ec3e802,1161.332132,374.153726,69 Jump Street,787.178405,155710.746098,200767.235367,Short
0xba939edf38c0ae0cc689c98b492e0535f43e4550,3624.343404,2967.269752,22Cap,657.073653,367248.974283,361264.057782,Short
0x3005fade4c0df5e1cd187d7062da359416f0eb8e,930.459938,361.383833,Delta_01,569.076106,23909.363797,3676.315250,Short
0xd2f03635901956b950737bbf02463dfad9f2e9e1,552.852172,6.859467,Gucky_1coin_1dot5x,545.992705,79625.574143,126693.452825,Short
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,305.942864,-191.661934,Hyperrr,497.604798,100545.085957,24558.486425,Short


,Parent PnL (USD),Cap PnL (USD),Vault,Parent minus cap PnL (USD),Parent buy turnover (USD),Cap buy turnover (USD),Period
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-2804.672708,-1013.303737,gaspedal,-1791.368972,24245.121704,10373.788123,Short
0xbbf7d7a9d0eaeab4115f022a6863450296112422,-2004.994935,-599.960861,Satori Quantum HF Vault,-1405.034074,406171.688578,168313.913575,Short
0x07fd993f0fa3a185f7207adccd29f7a87404689d,-647.664923,178.693886,[ Systemic Strategies ] L/S Grids,-826.358810,178756.170173,78907.404801,Short
0x45c42fbd450b5506f8dc819d46036630fe75b81e,-1794.812150,-977.145709,Octavious Maximus,-817.666441,40916.479009,28149.834137,Short
0x565b1c32936bdaa526947178acc95c5b08cfacc4,113.327101,853.185278,Mad Scientists,-739.858177,74619.074599,64616.381289,Short


,Parent PnL (USD),Cap PnL (USD),Vault,Parent minus cap PnL (USD),Parent buy turnover (USD),Cap buy turnover (USD),Period
0x3005fade4c0df5e1cd187d7062da359416f0eb8e,24920.079018,510.531167,Delta_01,24409.547850,299792.096022,36408.733331,Long
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,25183.629423,1573.879561,Realist Capital,23609.749862,425660.681868,84220.622759,Long
0xdfc24b077bc1425ad1dea75bcb6f8158e10df303,18005.061211,3923.201071,Hyperliquidity Provider (HLP),14081.860139,236137.077129,191131.766437,Long
0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,10691.371132,-66.987127,DOEZOE,10758.358260,41660.295043,952.718597,Long
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,8859.533105,58.425812,Hyperrr,8801.107293,179798.598274,26142.667425,Long
0x1e37a337ed460039d1b15bd3bc489de789768d5e,13597.909177,6071.336085,Growi HF,7526.573092,409784.775094,296406.078241,Long
0xb7ec2337fd25fd694959db750afe1cfefd33d2b7,5563.190065,197.283015,CryptoZer0_,5365.907050,39967.555039,30892.309117,Long
0x6fe9749eab7f66f002d7541d6c1e3bb3df19c701,4187.755372,144.157564,TA trader,4043.597809,33470.061582,3137.068179,Long
0x4dec0a851849056e259128464ef28ce78afa27f6,7808.704746,4290.868957,pmalt,3517.835789,358983.455289,160534.881841,Long
0xc1aff6f88f9acee8f3f27d45080281ca9d5526d4,3245.015725,332.831766,naze oppa degen nation,2912.183959,153741.215887,10405.031137,Long


,Parent PnL (USD),Cap PnL (USD),Vault,Parent minus cap PnL (USD),Parent buy turnover (USD),Cap buy turnover (USD),Period
0x6e4ccb8114c8157417b4ebb3ac8d62f8d2d70a25,-3374.285230,19.419929,cDOGE,-3393.705159,65388.766776,2580.753586,Long
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-3881.202140,-1021.289367,gaspedal,-2859.912773,38482.227535,10455.345898,Long
0x44ff912d0f88e27419ec0ddc950096609a9b6997,-1918.033801,620.887787,DailyTradeAI Low risk,-2538.921589,84078.281957,19955.448541,Long
0x04027058bb4aff889402ffb6e4743a96ce88149a,-982.536128,1438.128734,Road to 10 Million,-2420.664862,182680.147470,137388.599094,Long
0xbbf7d7a9d0eaeab4115f022a6863450296112422,-2916.585671,-604.796043,Satori Quantum HF Vault,-2311.789627,557536.861394,169670.351802,Long


,Run,Date,Side,Reserve (USD),Modelled redemption cost (USD),Position id
296,Short — v11 parent,2026-03-08,buy,16431.156214,0.000000,32
297,Short — v11 parent,2026-03-12,buy,7090.700817,0.000000,32
298,Short — v11 parent,2026-03-14,sell,7376.176010,7.383560,32
299,Short — v11 parent,2026-03-18,buy,1561.209403,0.000000,32
300,Short — v11 parent,2026-03-20,buy,855.580965,0.000000,32
301,Short — v11 parent,2026-03-24,buy,2692.807941,0.000000,32
302,Short — v11 parent,2026-03-30,sell,3077.199642,16.168012,32
303,Short — v11 parent,2026-04-01,buy,624.141202,0.000000,32
304,Short — v11 parent,2026-04-09,buy,1995.232725,0.000000,32
305,Short — v11 parent,2026-04-11,buy,12846.620159,0.000000,32


,Vault,Decision,Window observations,Exactly flat returns,Measured daily standard deviation,Source daily volatility floor,Daily sigma inferred from raw weight,Original requested dollars,Accepted dollars,TVL dollars
0,AJ Pro,2026-04-15,90,87,6.214130e-08,0.0001,0.0001,153743.83555,4348.835111,13178.288215


### Desired caps versus executed holdings

The dollar caps constrain desired targets. The unchanged sell band can leave actual holdings above a target when a trim is too small to execute. The audit below matches post-trade position snapshots to the allocation ledger, excludes already closed positions, and reports this execution tolerance separately from sizing invariants. Its sell-band approximation uses the fresh budget divided by the 98% deployment fraction; the live decision threshold uses pre-fee gross reference equity.


In [13]:
import json
from pathlib import Path
import pandas as pd
p=ARTIFACT_DIR
summary=[]
for period in ['short','long']:
 s=json.loads((p/f'{period}-accepted_dollar_cap-state.json').read_text())
 positions={**s['portfolio']['open_positions'],**s['portfolio']['closed_positions'],**s['portfolio']['frozen_positions']}
 samples=[]
 for pid,snaps in s['stats']['positions'].items():
  pos=positions[pid]
  for snap in snaps:
   ts=snap['calculated_at']
   if pos['closed_at'] is not None and pos['closed_at']<=ts:continue
   samples.append({'Date':pd.Timestamp(ts,unit='s'),'Pair id':pos['pair']['internal_id'],'Actual holding (USD)':snap['value']})
 actual=pd.DataFrame(samples).groupby(['Date','Pair id'],as_index=False)['Actual holding (USD)'].sum()
 l=pd.read_csv(p/f'{period}-accepted_dollar_cap-allocations.csv',parse_dates=['Date'])
 m=l.merge(actual,on=['Date','Pair id'],how='left')
 m['Actual holding (USD)']=m['Actual holding (USD)'].fillna(0)
 m['Excess target (USD)']=(m['Actual holding (USD)']-m['Final target (USD)']).clip(lower=0)
 m['Buy minimum (USD)']=m['Deployable budget (USD)']/.98*.0005
 m['Approximate sell band (USD)']=m['Deployable budget (USD)']/.98*.005
 excess=m[m['Excess target (USD)']>1]
 m.to_csv(p/f'{period}-executed-holdings-audit.csv',index=False)
 summary.append({'Period':period,'Rows with post-trade holdings above target by >$1':len(excess),'Largest excess (USD)':excess['Excess target (USD)'].max(),'Largest aggregate excess per cycle (USD)':m.groupby('Date')['Excess target (USD)'].sum().max(),'Rows above approximate sell band':int((excess['Excess target (USD)']>excess['Approximate sell band (USD)']+1).sum())})
 display(excess.nlargest(3,'Excess target (USD)')[['Date','Vault','Final target (USD)','Actual holding (USD)','Excess target (USD)','Approximate sell band (USD)','Flags']])
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(p/'executed-holdings-summary.csv',index=False)


,Date,Vault,Final target (USD),Actual holding (USD),Excess target (USD),Approximate sell band (USD),Flags
901,2026-09-28,Buyback Spread Engine,991.607241,1792.940646,801.333405,801.826632,individual_trade_size_too_small
280,2026-03-26,pmalt,22283.629437,23066.890368,783.260931,792.553122,individual_trade_size_too_small
821,2026-09-04,Sequoia HyperStable Yield Optimizer,499.828367,1283.080846,783.252478,812.041085,individual_trade_size_too_small


,Date,Vault,Final target (USD),Actual holding (USD),Excess target (USD),Approximate sell band (USD),Flags
2892,2026-09-28,Buyback Spread Engine,999.588801,1807.379904,807.791103,808.280626,individual_trade_size_too_small
2271,2026-03-26,pmalt,22457.210703,23247.063017,789.852314,798.726819,individual_trade_size_too_small
2812,2026-09-04,Sequoia HyperStable Yield Optimizer,503.859177,1293.428623,789.569446,818.589699,individual_trade_size_too_small


,Period,Rows with post-trade holdings above target by >$1,Largest excess (USD),Largest aggregate excess per cycle (USD),Rows above approximate sell band
0,short,262,801.333405,2050.788823,0
1,long,746,807.791103,2065.380793,0
